In [1]:
import requests
import datetime
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor
from collections import deque
import threading

In [2]:
# API Key 
API_KEY = 'ksnxCfuMd8YcYtNVDrqZKBY0aZeDMLX8'

In [3]:
# API Rate Limit Constants
API_LIMIT = 300  # Max API calls per minute
WINDOW_SIZE = 60  # Seconds (1 minute)
MAX_RETRIES = 3   # Maximum retry attempts
BACKOFF_FACTOR = 2  # Exponential backoff factor

# Track API request timestamps
request_times = deque()
request_lock = threading.Lock()  # Prevents race conditions when updating the queue

In [4]:
def rate_limit():
    """Strictly enforces the 300 API calls per minute limit."""
    global request_times

    with request_lock:
        now = time.time()

        # Remove timestamps older than 60 seconds
        while request_times and now - request_times[0] > WINDOW_SIZE:
            request_times.popleft()

        # If we're at the limit, wait until an API slot opens
        while len(request_times) >= API_LIMIT:
            wait_time = WINDOW_SIZE - (now - request_times[0])
            print(f"Rate limit reached! Sleeping for {wait_time:.2f} seconds...")
            time.sleep(wait_time)

            # Remove old timestamps and check again
            now = time.time()
            while request_times and now - request_times[0] > WINDOW_SIZE:
                request_times.popleft()

        # Register the new API request timestamp
        request_times.append(time.time())


def fetch_data_with_retries(url):
    """Fetches data from API with automatic retries and strict rate limiting."""
    for attempt in range(1, MAX_RETRIES + 1):
        rate_limit()  # Ensure we respect API limits before making a request

        try:
            response = requests.get(url)
            if response.status_code == 200:
                return response.json()
            else:
                print(f"Attempt {attempt}: API returned {response.status_code}. Retrying...")
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt}: Network error: {e}. Retrying...")

        # Exponential backoff before retrying
        time.sleep(BACKOFF_FACTOR ** attempt)

    print(f"Failed after {MAX_RETRIES} attempts: {url}")
    return None


def get_all_tickers():
    """Fetches a list of all investable stocks and indexes from FMP API."""
    stock_url = f"https://financialmodelingprep.com/api/v3/stock/list?apikey={API_KEY}"
    index_url = f"https://financialmodelingprep.com/api/v3/symbol/available-indexes?apikey={API_KEY}"

    stock_response = fetch_data_with_retries(stock_url)
    index_response = fetch_data_with_retries(index_url)

    if not stock_response or not index_response:
        print("Failed to retrieve stock/index lists.")
        return []

    stock_tickers = [item['symbol'] for item in stock_response]
    index_tickers = [item['symbol'] for item in index_response]

    return stock_tickers + index_tickers


def get_sd(ticker):
    """Fetches standard deviation for a stock over 1Y, 5Y, and 10Y."""
    base_url = 'https://financialmodelingprep.com/api/v3/technical_indicator/1day/'
    periods = {"1Y": 252, "5Y": 1260, "10Y": 2520}
    sd_values = {}

    for key, period in periods.items():
        url = f"{base_url}{ticker}?type=standardDeviation&period={period}&apikey={API_KEY}"
        data = fetch_data_with_retries(url)

        if isinstance(data, list) and len(data) > 0:
            sd_values[key] = round(data[0].get('standardDeviation', None), 4)
        else:
            sd_values[key] = None

    return sd_values


def get_cagr(ticker):
    """Fetches CAGR for a stock over 1Y, 5Y, and 10Y."""
    base_url = 'https://financialmodelingprep.com/api/v3/historical-price-full/'
    years_list = [1, 5, 10]
    cagr_values = {}

    for years in years_list:
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=years * 365)

        url = f"{base_url}{ticker}?from={start_date}&to={end_date}&apikey={API_KEY}"
        data = fetch_data_with_retries(url)

        if data and "historical" in data and len(data["historical"]) > 0:
            historical_data = sorted(data["historical"], key=lambda x: x["date"])
            P_start = historical_data[0]["close"]
            P_end = historical_data[-1]["close"]

            cagr = ((P_end / P_start) ** (1 / years)) - 1
            cagr_values[f"{years}Y"] = round(cagr * 100, 2)
        else:
            cagr_values[f"{years}Y"] = None

    return cagr_values


def get_stock_data(ticker):
    """Combines Standard Deviation and CAGR data into a single dictionary per stock."""
    try:
        sd_data = get_sd(ticker)
        cagr_data = get_cagr(ticker)

        return {
            "Ticker": ticker,
            "1Y SD": sd_data["1Y"],
            "5Y SD": sd_data["5Y"],
            "10Y SD": sd_data["10Y"],
            "1Y CAGR": cagr_data["1Y"],
            "5Y CAGR": cagr_data["5Y"],
            "10Y CAGR": cagr_data["10Y"]
        }
    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        return None


def get_multiple_stocks_data(tickers, workers=20):
    """Fetches SD and CAGR for multiple stocks using threading while respecting rate limits."""
    total_tickers = len(tickers)
    all_data = []

    print(f"Processing {total_tickers} tickers using {workers} threads...")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = list(executor.map(get_stock_data, tickers))

    # Filter out None values (failed requests)
    all_data = [result for result in results if result]

    # Convert to DataFrame
    df = pd.DataFrame(all_data)

    # Save interim results
    df.to_csv("all_stocks_sd_cagr.csv", index=False)

    return df

In [5]:
# **Step 1: Get All Investable Stocks & Indexes**
all_tickers = get_all_tickers()

# **Step 2: Process API Calls in Parallel with Strict Rate Limiting**
df = get_multiple_stocks_data(all_tickers, workers=20)

# **Step 3: Save Final Data**
df.to_csv("all_stocks_sd_cagr.csv", index=False)

# **Step 4: Display Sample Output**
print(df.head())

Processing 85099 tickers using 20 threads...
Rate limit reached! Sleeping for 28.77 seconds...
Rate limit reached! Sleeping for 5.12 seconds...
Rate limit reached! Sleeping for 0.68 seconds...
Rate limit reached! Sleeping for 0.41 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.03 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.59 seconds...
Rate limit reached! Sleeping for 0.02 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.21 seconds...
Rate limit reached! Sleeping for 0.72 seconds...
Rate limit reached! Sleeping for 0.16 seconds...
Rate limit reached! Sle

Rate limit reached! Sleeping for 0.32 seconds...
Rate limit reached! Sleeping for 0.20 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.18 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.08 seconds...
Rate limit reached! Sleeping for 0.40 seconds...
Rate limit reached! Sleeping for 0.08 seconds...
Rate limit reached! Sleeping for 0.07 seconds...
Rate limit reached! Sleeping for 0.27 seconds...
Rate limit reached! Sleeping for 0.04 seconds...
Rate limit reached! Sleeping for 0.08 seconds...
Rate limit reached! Sleeping for 0.05 seconds...
Rate limit reached! Sleeping for 0.02 seconds...
Rate limit reached! Sleeping for 0.09 seconds...
Rate limit reached! Sleeping for 0.03 seconds...
Rate limit reached! Sleeping for 0.04 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.09 seconds...
Rate limit reached! Sleeping for 0.05 seconds...
Rate limit reached! 

Rate limit reached! Sleeping for 0.07 seconds...
Rate limit reached! Sleeping for 0.15 seconds...
Rate limit reached! Sleeping for 0.16 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.10 seconds...
Rate limit reached! Sleeping for 0.06 seconds...
Rate limit reached! Sleeping for 0.06 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.04 seconds...
Rate limit reached! Sleeping for 0.04 seconds...
Rate limit reached! Sleeping for 0.12 seconds...
Rate limit reached! Sleeping for 0.25 seconds...
Rate limit reached! Sleeping for 0.05 seconds...
Rate limit reached! Sleeping for 0.02 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.07 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.02 seconds...
Rate limit reached! Sleeping for 0.08 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! 

Rate limit reached! Sleeping for 0.03 seconds...
Rate limit reached! Sleeping for 0.08 seconds...
Rate limit reached! Sleeping for 0.07 seconds...
Rate limit reached! Sleeping for 0.03 seconds...
Rate limit reached! Sleeping for 0.07 seconds...
Rate limit reached! Sleeping for 0.03 seconds...
Rate limit reached! Sleeping for 0.04 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.12 seconds...
Rate limit reached! Sleeping for 0.13 seconds...
Rate limit reached! Sleeping for 0.13 seconds...
Rate limit reached! Sleeping for 0.08 seconds...
Rate limit reached! Sleeping for 0.02 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.09 seconds...
Rate limit reached! Sleeping for 0.17 seconds...
Rate limit reached! Sleeping for 0.19 seconds...
Rate limit reached! Sleeping for 0.02 seconds...
Rate limit reached! Sleeping for 0.15 seconds...
Rate limit reached! Sleeping for 0.12 seconds...
Rate limit reached! 

Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.09 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.05 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.03 seconds...
Rate limit reached! Sleeping for 0.02 seconds...
Rate limit reached! Sleeping for 0.07 seconds...
Rate limit reached! Sleeping for 0.03 seconds...
Rate limit reached! Sleeping for 0.22 seconds...
Rate limit reached! Sleeping for 0.08 seconds...
Rate limit reached! Sleeping for 0.08 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.02 seconds...
Rate limit reached! Sleeping for 0.11 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.04 seconds...
Rate limit reached! Sleeping for 0.04 seconds...
Rate limit reached! 

...

Rate limit reached! Sleeping for 0.21 seconds...
Rate limit reached! Sleeping for 0.09 seconds...
Rate limit reached! Sleeping for 0.11 seconds...
Rate limit reached! Sleeping for 0.78 seconds...
Rate limit reached! Sleeping for 0.42 seconds...
Rate limit reached! Sleeping for 0.17 seconds...
Rate limit reached! Sleeping for 0.34 seconds...
Rate limit reached! Sleeping for 0.49 seconds...
Rate limit reached! Sleeping for 0.13 seconds...
Rate limit reached! Sleeping for 0.05 seconds...
Rate limit reached! Sleeping for 0.04 seconds...
Rate limit reached! Sleeping for 0.11 seconds...
Rate limit reached! Sleeping for 0.05 seconds...
Rate limit reached! Sleeping for 0.05 seconds...
Rate limit reached! Sleeping for 0.02 seconds...
Rate limit reached! Sleeping for 0.16 seconds...
Rate limit reached! Sleeping for 0.27 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.07 seconds...
Rate limit reached! 

Rate limit reached! Sleeping for 0.21 seconds...
Rate limit reached! Sleeping for 0.23 seconds...
Rate limit reached! Sleeping for 0.23 seconds...
Rate limit reached! Sleeping for 0.30 seconds...
Rate limit reached! Sleeping for 0.50 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.02 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.01 seconds...
Rate limit reached! Sleeping for 0.49 seconds...
Rate limit reached! Sleeping for 0.21 seconds...
Rate limit reached! Sleeping for 0.10 seconds...
Rate limit reached! Sleeping for 0.12 seconds...
Rate limit reached! Sleeping for 0.79 seconds...
Rate limit reached! Sleeping for 0.42 seconds...
Rate limit reached! Sleeping for 0.17 seconds...
Rate limit reached! Sleeping for 0.34 seconds...
Rate limit reached! Sleeping for 0.48 seconds...
Rate limit reached! Sleeping for 0.14 seconds...
Rate limit reached! Sleeping for 0.05 seconds...
Rate limit reached! 

Rate limit reached! Sleeping for 0.12 seconds...
Rate limit reached! Sleeping for 0.17 seconds...
Rate limit reached! Sleeping for 0.21 seconds...
Rate limit reached! Sleeping for 0.25 seconds...
Rate limit reached! Sleeping for 0.22 seconds...
Rate limit reached! Sleeping for 0.25 seconds...
Rate limit reached! Sleeping for 0.48 seconds...
Rate limit reached! Sleeping for 0.03 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.48 seconds...
Rate limit reached! Sleeping for 0.20 seconds...
Rate limit reached! Sleeping for 0.10 seconds...
Rate limit reached! Sleeping for 0.12 seconds...
Rate limit reached! Sleeping for 0.79 seconds...
Rate limit reached! Sleeping for 0.43 seconds...
Rate limit reached! Sleeping for 0.18 seconds...
Rate limit reached! Sleeping for 0.35 seconds...
Rate limit reached! Sleeping for 0.48 seconds...
Rate limit reached! Sleeping for 0.15 seconds...
Rate limit reached! 

Rate limit reached! Sleeping for 0.14 seconds...
Rate limit reached! Sleeping for 0.06 seconds...
Rate limit reached! Sleeping for 0.10 seconds...
Rate limit reached! Sleeping for 0.02 seconds...
Rate limit reached! Sleeping for 0.03 seconds...
Rate limit reached! Sleeping for 0.03 seconds...
Rate limit reached! Sleeping for 0.11 seconds...
Rate limit reached! Sleeping for 0.16 seconds...
Rate limit reached! Sleeping for 0.21 seconds...
Rate limit reached! Sleeping for 0.25 seconds...
Rate limit reached! Sleeping for 0.23 seconds...
Rate limit reached! Sleeping for 0.29 seconds...
Rate limit reached! Sleeping for 0.49 seconds...
Rate limit reached! Sleeping for 0.03 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.00 seconds...
Rate limit reached! Sleeping for 0.49 seconds...
Rate limit reached! Sleeping for 0.20 seconds...
Rate limit reached! Sleeping for 0.08 seconds...
Rate limit reached! Sleeping for 0.11 seconds...
Rate limit reached! 

KeyboardInterrupt: 